In [1]:
import os
import glob
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np

In [2]:
def evaluate_land_accuracy_cropped_ref(ref_tiff, pred_tiff, ref_nodata_value=None, pred_nodata_value=None):
    """
    Evaluates land mask accuracy by cropping/reprojecting a larger reference mask 
    to match a smaller predicted mask. NoData represents Water, valid data represents Land.
    """
    
    with rasterio.open(ref_tiff) as src_ref, rasterio.open(pred_tiff) as src_pred:
        
        # 1. Read the smaller predicted image (this sets our target grid)
        pred_data = src_pred.read(1)
        
        # 2. Create an empty array to hold the cropped/reprojected reference data
        # It perfectly matches the dimensions of the predicted data
        ref_data_reproj = np.empty_like(pred_data)
        
        print(f"Cropping and reprojecting Reference ({src_ref.crs}) to match Predicted ({src_pred.crs})...")
        
        # 3. Reproject the larger reference raster into the smaller predicted grid
        reproject(
            source=rasterio.band(src_ref, 1),
            destination=ref_data_reproj,
            src_transform=src_ref.transform,
            src_crs=src_ref.crs,
            dst_transform=src_pred.transform,
            dst_crs=src_pred.crs,
            resampling=Resampling.nearest 
        )
        
        # --- Evaluate Land Accuracy ---
        pred_flat = pred_data.flatten()
        ref_flat = ref_data_reproj.flatten()
        
        # Identify NoData values
        ref_nodata = ref_nodata_value if ref_nodata_value is not None else src_ref.nodata
        pred_nodata = pred_nodata_value if pred_nodata_value is not None else src_pred.nodata
        
        if ref_nodata is None or pred_nodata is None:
            raise ValueError("NoData values must be defined in the raster metadata or passed as arguments.")

        # Define Land as True (Not NoData) and Water as False (NoData)
        actual_land = (ref_flat != ref_nodata)
        predicted_land = (pred_flat != pred_nodata)
        
        # Calculate True Positives, False Positives, False Negatives
        true_positives = np.sum(actual_land & predicted_land)     # Actual Land predicted as Land
        false_positives = np.sum(~actual_land & predicted_land)   # Actual Water predicted as Land
        false_negatives = np.sum(actual_land & ~predicted_land)   # Actual Land predicted as Water
        
        total_actual_land = np.sum(actual_land)
        total_predicted_land = np.sum(predicted_land)
        
        # Calculate Metrics
        recall = true_positives / total_actual_land if total_actual_land > 0 else 0.0
        precision = true_positives / total_predicted_land if total_predicted_land > 0 else 0.0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        iou = true_positives / (true_positives + false_positives + false_negatives) if (true_positives + false_positives + false_negatives) > 0 else 0.0
        
        print("\n--- Land Accuracy Report ---")
        print(f"Total Actual Land Pixels:    {total_actual_land:,}")
        print(f"Total Predicted Land Pixels: {total_predicted_land:,}")
        print(f"Correctly Predicted Land:    {true_positives:,}")
        print("-" * 40)
        print(f"Recall (Producer's Accuracy): {recall:.4f}  <- % of actual land found")
        print(f"Precision (User's Accuracy):  {precision:.4f}  <- % of predicted land that is correct")
        print(f"F1-Score:                     {f1_score:.4f}")
        print(f"IoU (Intersection over Union):{iou:.4f}")

In [7]:
# Example usage
# The reference image acts as the base map (target CRS and target grid)
reference_file = "/Users/sahuang/Documents/DATA/am_samoa/Landcover/am_samoa_ccap_ocean_mask_small.tif"

# The predicted image will be reprojected, cropped, and resampled to match the reference
predicted_file = "/Users/sahuang/Documents/DATA/umbra/aunuu/Aunuu_RTC/20250210_filt_6_cluster_smoothed_mask_morph.tif"

# Run evaluation (assumes 255 represents NoData)
# evaluate_land_accuracy(reference_file, predicted_file, nodata_value=0)

 # You can pass different NoData values if the two files use different markers for water
evaluate_land_accuracy_cropped_ref(
        reference_file, 
        predicted_file, 
        ref_nodata_value=0, 
        pred_nodata_value=0
    )

Cropping and reprojecting Reference (EPSG:32702) to match Predicted (EPSG:4326)...

--- Land Accuracy Report ---
Total Actual Land Pixels:    9,599,972
Total Predicted Land Pixels: 3,406,369
Correctly Predicted Land:    3,330,943
----------------------------------------
Recall (Producer's Accuracy): 0.3470  <- % of actual land found
Precision (User's Accuracy):  0.9779  <- % of predicted land that is correct
F1-Score:                     0.5122
IoU (Intersection over Union):0.3443


In [5]:
filepath = '/Users/sahuang/Documents/DATA/umbra/ALL_RTC_INPUT/'
filenames = os.listdir(filepath)
suffix = '_cluster_smoothed_mask_morph_auto.tif'

reference_file = "/Users/sahuang/Documents/DATA/am_samoa/Landcover/am_samoa_ccap_ocean_mask_small.tif"

for filename in filenames:
    if suffix in filename:
        predicted_file = filepath + filename 
        evaluate_land_accuracy_cropped_ref(
            reference_file, 
            predicted_file, 
            ref_nodata_value=0, 
            pred_nodata_value=0
            )

Cropping and reprojecting Reference (EPSG:32702) to match Predicted (EPSG:4326)...

--- Land Accuracy Report ---
Total Actual Land Pixels:    34,430,628
Total Predicted Land Pixels: 6,125,903
Correctly Predicted Land:    5,957,298
----------------------------------------
Recall (Producer's Accuracy): 0.1730  <- % of actual land found
Precision (User's Accuracy):  0.9725  <- % of predicted land that is correct
F1-Score:                     0.2938
IoU (Intersection over Union):0.1722
Cropping and reprojecting Reference (EPSG:32702) to match Predicted (EPSG:4326)...

--- Land Accuracy Report ---
Total Actual Land Pixels:    7,889,375
Total Predicted Land Pixels: 4,361,274
Correctly Predicted Land:    4,242,577
----------------------------------------
Recall (Producer's Accuracy): 0.5378  <- % of actual land found
Precision (User's Accuracy):  0.9728  <- % of predicted land that is correct
F1-Score:                     0.6926
IoU (Intersection over Union):0.5298
Cropping and reprojecting R

In [9]:
def evaluate_directory_composite(ref_tiff, pred_dir, suffix, ref_nodata_value=None, pred_nodata_value=None):
    """
    Evaluates all predicted TIFFs in a directory against a single large reference TIFF.
    Calculates composite Land accuracy statistics across all images.
    """
    
    # Find all TIFF files in the predicted directory
    pred_files = glob.glob(os.path.join(pred_dir, "*" + suffix))
    
    if not pred_files:
        print(f"No .tif files found in directory: {pred_dir}")
        return

    print(f"Found {len(pred_files)} images to process. Calculating composite statistics...\n")

    # Initialize running totals
    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_actual = 0
    total_predicted = 0

    # Open the large reference image once
    with rasterio.open(ref_tiff) as src_ref:
        ref_nodata = ref_nodata_value if ref_nodata_value is not None else src_ref.nodata
        
        if ref_nodata is None:
             raise ValueError("Reference NoData value must be defined.")

        for idx, pred_file in enumerate(pred_files, 1):
            print(f"Processing {idx}/{len(pred_files)}: {os.path.basename(pred_file)}")
            
            with rasterio.open(pred_file) as src_pred:
                pred_nodata = pred_nodata_value if pred_nodata_value is not None else src_pred.nodata
                
                if pred_nodata is None:
                    print(f"  Skipping {os.path.basename(pred_file)}: NoData value undefined.")
                    continue
                
                # Read predicted data
                pred_data = src_pred.read(1)
                
                # Create empty array for reprojected reference data
                ref_data_reproj = np.empty_like(pred_data)
                
                # Reproject reference to match the current predicted image
                reproject(
                    source=rasterio.band(src_ref, 1),
                    destination=ref_data_reproj,
                    src_transform=src_ref.transform,
                    src_crs=src_ref.crs,
                    dst_transform=src_pred.transform,
                    dst_crs=src_pred.crs,
                    resampling=Resampling.nearest 
                )
                
                # Flatten arrays
                pred_flat = pred_data.flatten()
                ref_flat = ref_data_reproj.flatten()
                
                # Define Land as True (Not NoData) and Water as False (NoData)
                actual_land = (ref_flat != ref_nodata)
                predicted_land = (pred_flat != pred_nodata)
                
                # Accumulate counts for this image
                total_actual += np.sum(actual_land)
                total_predicted += np.sum(predicted_land)
                
                total_tp += np.sum(actual_land & predicted_land)
                total_fp += np.sum(~actual_land & predicted_land)
                total_fn += np.sum(actual_land & ~predicted_land)

    # --- Calculate Composite Metrics ---
    print("\n" + "="*50)
    print("      COMPOSITE LAND ACCURACY REPORT")
    print("="*50)
    
    if total_actual == 0 and total_predicted == 0:
        print("No land pixels found in any of the images.")
        return

    # Calculate final metrics using aggregated counts
    recall = total_tp / total_actual if total_actual > 0 else 0.0
    precision = total_tp / total_predicted if total_predicted > 0 else 0.0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    iou = total_tp / (total_tp + total_fp + total_fn) if (total_tp + total_fp + total_fn) > 0 else 0.0
    
    print(f"Total Images Processed:      {len(pred_files)}")
    print(f"Total Actual Land Pixels:    {total_actual:,}")
    print(f"Total Predicted Land Pixels: {total_predicted:,}")
    print(f"Total Correctly Predicted:   {total_tp:,}")
    print("-" * 50)
    print(f"Composite Recall (Producer's): {recall:.4f}")
    print(f"Composite Precision (User's):  {precision:.4f}")
    print(f"Composite F1-Score:            {f1_score:.4f}")
    print(f"Composite IoU:                 {iou:.4f}")
    print("="*50)

In [10]:
predicted_directory = '/Users/sahuang/Documents/DATA/umbra/ALL_RTC_INPUT/'
reference_file = "/Users/sahuang/Documents/DATA/am_samoa/Landcover/am_samoa_ccap_ocean_mask_small.tif"

# Run the composite evaluation
evaluate_directory_composite(
    ref_tiff=reference_file, 
    pred_dir=predicted_directory, 
    suffix='_cluster_smoothed_mask_morph_auto.tif',
    ref_nodata_value=0, 
    pred_nodata_value=0
)

Found 25 images to process. Calculating composite statistics...

Processing 1/25: 20250429_filt_5_cluster_smoothed_mask_morph_auto.tif
Processing 2/25: 20250611_filt_4_cluster_smoothed_mask_morph_auto.tif
Processing 3/25: 20250210_filt_5_cluster_smoothed_mask_morph_auto.tif
Processing 4/25: 20241218_filt_4_cluster_smoothed_mask_morph_auto.tif
Processing 5/25: Ofu_20241219_filt_4_cluster_smoothed_mask_morph_auto.tif
Processing 6/25: 20250321_filt_5_cluster_smoothed_mask_morph_auto.tif
Processing 7/25: 20250525_filt_4_cluster_smoothed_mask_morph_auto.tif
Processing 8/25: 20241218_filt_8_cluster_smoothed_mask_morph_auto.tif
Processing 9/25: 20250705_filt_4_cluster_smoothed_mask_morph_auto.tif
Processing 10/25: Ofu_20241224_filt_4_cluster_smoothed_mask_morph_auto.tif
Processing 11/25: 20250511_filt_4_cluster_smoothed_mask_morph_auto.tif
Processing 12/25: 20250403_filt_4_cluster_smoothed_mask_morph_auto.tif
Processing 13/25: 20250623_filt_5_cluster_smoothed_mask_morph_auto.tif
Processing 14

In [ ]:
# predicted_directory = '/Users/sahuang/Documents/DATA/umbra/ALL_RTC_INPUT/'
predicted_directory = '/Users/sahuang/Documents/DATA/umbra/ALL_SHORELINES_MANUAL/'
reference_file = "/Users/sahuang/Documents/DATA/am_samoa/Landcover/am_samoa_ccap_ocean_mask_small.tif"

# Run the composite evaluation
evaluate_directory_composite(
    ref_tiff=reference_file, 
    pred_dir=predicted_directory, 
    suffix="_mask_morph.tif",
    ref_nodata_value=0, 
    pred_nodata_value=0
)